In [36]:
#Code có thư viện
#Import các thư viện cần thiết và tải dữ liệu

import pandas as pd

pd.set_option('display.max_rows', None)
df = pd.read_csv("Housing.csv")

print(df)

        price   area  bedrooms  bathrooms  stories mainroad guestroom  \
0    13300000   7420         4          2        3      yes        no   
1    12250000   8960         4          4        4      yes        no   
2    12250000   9960         3          2        2      yes        no   
3    12215000   7500         4          2        2      yes        no   
4    11410000   7420         4          1        2      yes       yes   
5    10850000   7500         3          3        1      yes        no   
6    10150000   8580         4          3        4      yes        no   
7    10150000  16200         5          3        2      yes        no   
8     9870000   8100         4          1        2      yes       yes   
9     9800000   5750         3          2        4      yes       yes   
10    9800000  13200         3          1        2      yes        no   
11    9681000   6000         4          3        2      yes       yes   
12    9310000   6550         4          2        2 

In [37]:
#1. Xử lý dữ liệu thiếu
df['price'].fillna(df['price'].mean(), inplace=True)
df['area'].fillna(df['area'].mean(), inplace=True)
df['bedrooms'].fillna(df['bedrooms'].mean(), inplace=True)
df['bathrooms'].fillna(df['bathrooms'].mean(), inplace=True)
df['stories'].fillna(df['stories'].mean(), inplace=True)
df['mainroad'].fillna(df['mainroad'].mode()[0], inplace=True)
df['guestroom'].fillna(df['guestroom'].mode()[0], inplace=True)
df['basement'].fillna(df['basement'].mode()[0], inplace=True)
df['hotwaterheating'].fillna(df['hotwaterheating'].mode()[0], inplace=True)
df['airconditioning'].fillna(df['airconditioning'].mode()[0], inplace=True)
df['parking'].fillna(df['parking'].mean(), inplace=True)
df['prefarea'].fillna(df['prefarea'].mode()[0], inplace=True)
df['furnishingstatus'].fillna(df['furnishingstatus'].mode()[0], inplace=True)


print(df.isnull().sum())

price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64


In [38]:
#2. Mã hóa biến phân loại bằng One-Hot Encoding

df = pd.get_dummies(df, columns=['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus'], drop_first=True)

print(df.head())

      price  area  bedrooms  bathrooms  stories  parking  mainroad_yes  \
0  13300000  7420         4          2        3        2             1   
1  12250000  8960         4          4        4        3             1   
2  12250000  9960         3          2        2        2             1   
3  12215000  7500         4          2        2        3             1   
4  11410000  7420         4          1        2        2             1   

   guestroom_yes  basement_yes  hotwaterheating_yes  airconditioning_yes  \
0              0             0                    0                    1   
1              0             0                    0                    1   
2              0             1                    0                    0   
3              0             1                    0                    1   
4              1             1                    0                    1   

   prefarea_yes  furnishingstatus_semi-furnished  furnishingstatus_unfurnished  
0             1  

In [39]:
#3. Tạo đặc trưng mới

df['PricePerArea'] = df['price'] / df['area']
df.head()

,price,area,bedrooms,bathrooms,stories,parking,mainroad_yes,guestroom_yes,basement_yes,hotwaterheating_yes,airconditioning_yes,prefarea_yes,furnishingstatus_semi-furnished,furnishingstatus_unfurnished,PricePerArea
0,13300000,7420,4,2,3,2,1,0,0,0,1,1,0,0,1792.452830
1,12250000,8960,4,4,4,3,1,0,0,0,1,0,0,0,1367.187500
2,12250000,9960,3,2,2,2,1,0,1,0,0,1,1,0,1229.919679
3,12215000,7500,4,2,2,3,1,0,1,0,1,1,0,0,1628.666667
4,11410000,7420,4,1,2,2,1,1,1,0,1,0,0,0,1537.735849


In [40]:
#4: Chuẩn hóa dữ liệu

numerical_features = ['price', 'area', 'bedrooms', 'bathrooms', 'parking', 'PricePerArea']

scaler = StandardScaler()
df[numerical_features] = scaler.fit_transform(df[numerical_features])


print(df.head())

      price      area  bedrooms  bathrooms  stories   parking  mainroad_yes  \
0  4.566365  1.046726  1.403419   1.421812        3  1.517692             1   
1  4.004484  1.757010  1.403419   5.405809        4  2.679409             1   
2  4.004484  2.218232  0.047278   1.421812        2  1.517692             1   
3  3.985755  1.083624  1.403419   1.421812        2  2.679409             1   
4  3.554979  1.046726  1.403419  -0.570187        2  1.517692             1   

   guestroom_yes  basement_yes  hotwaterheating_yes  airconditioning_yes  \
0              0             0                    0                    1   
1              0             0                    0                    1   
2              0             1                    0                    0   
3              0             1                    0                    1   
4              1             1                    0                    1   

   prefarea_yes  furnishingstatus_semi-furnished  \
0             1 

In [41]:
#Bổ sung: Xây dựng mô hình và đánh giá
X = df.drop(columns=['hotwaterheating_yes'])
y = df['stories']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy of the model: {accuracy * 100:.2f}%")

Accuracy of the model: 97.25%
